# Scenario 8: Agent Control — Full-Lifecycle Agent Monitoring with Galileo

---

## What You'll Learn

In this notebook, you will learn how to:

1. **Observe** agent decisions with structured spans (agent, tool, workflow, LLM)
2. **Evaluate** agent quality with agentic metrics (9 metrics covering efficiency, flow, tool usage, and reasoning)
3. **Protect** agents at runtime with guardrails built on agentic metric rules

---

## What is Agent Control?

**Agent Control** is the combination of three pillars for managing AI agents in production:

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   OBSERVE   │ ──→ │  EVALUATE   │ ──→ │   PROTECT   │
│  Log spans  │     │   Score     │     │  Guardrails │
│  & traces   │     │   quality   │     │  at runtime │
└─────────────┘     └─────────────┘     └─────────────┘
```

- **Observe** — Log every agent decision, tool call, and LLM interaction as structured spans
- **Evaluate** — Score agent behavior with metrics like efficiency, flow coherence, and tool selection quality
- **Protect** — Apply runtime guardrails that block or override unsafe agent outputs

---

## Key Concepts

| Concept | What It Means |
|---|---|
| **Agent Control** | The three-pillar approach: Observe + Evaluate + Protect for AI agents |
| **Agentic Metrics** | 9 metrics for measuring agent quality (efficiency, flow, tool usage, reasoning, etc.) |
| **Agent Guardrails** | Runtime protection using agentic metric rules to block unsafe agent behavior |
| **Circular Tool Detection** | A pattern for preventing infinite loops when agents repeatedly call the same tools |
| **Agent Framework Integration** | Auto-instrumentation for agent frameworks using the `@log` decorator |

---

> **Note:** This notebook uses **manual span logging** (no real LLM calls) to demonstrate the full agent control lifecycle. The data is simulated so you can focus on learning the three pillars without needing OpenAI API credits.

---

## Prerequisites

- Galileo API key set in your `.env` file
- The `galileo` Python package installed (`uv sync` from the repo root)
- Completion of Scenario 3 (Tools) and Scenario 5 (Guardrails) is recommended but not required

### Step 0: Load Environment Variables

This cell loads your `.env` file, which contains your `GALILEO_API_KEY` (and optionally `OPENAI_API_KEY`). The code searches for the `.env` file in the current directory and the parent directory so the notebook works regardless of where you run it from.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

### Step 0b: Import Libraries

Here we import everything needed for the three pillars of Agent Control:

**Observe (logging):**
- `GalileoLogger` — manual logger for creating traces with agent, tool, workflow, and LLM spans
- `Message` / `MessageRole` — data classes for representing chat messages
- `galileo_context` — global context manager for connecting to a Galileo project
- `galileo_log` (decorator) — automatically wraps Python functions as Galileo spans

**Evaluate (metrics):**
- `GalileoMetrics` — enum of all available metric types
- `enable_metrics` — turns on metric scoring for a log stream

**Protect (guardrails):**
- `Payload` / `Ruleset` — wrap text and define rules for guardrail stages
- `create_protect_stage` / `get_protect_stage` / `invoke_protect` / `update_protect_stage` — manage and invoke guardrail stages
- `Rule` / `RuleOperator` / `OverrideAction` / `StageType` — lower-level schemas for defining guardrail behavior

In [ ]:
from galileo import (
    GalileoLogger, GalileoMetrics, Message, MessageRole,
    Payload, Ruleset, galileo_context,
    create_protect_stage, get_protect_stage, invoke_protect, update_protect_stage,
)
from galileo.config import GalileoPythonConfig
from galileo.decorator import log as galileo_log
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.projects import delete_project
from galileo_core.schemas.protect.action import OverrideAction
from galileo_core.schemas.protect.rule import Rule, RuleOperator
from galileo_core.schemas.protect.stage import StageType
from galileo.exceptions import NotFoundError

PROJECT_NAME = 'GalileoEval_S8_AgentControl'
LOG_STREAM_NAME = 'agent-control'
MODEL = 'gpt-4o-mini'

## Step 1: Initialize Galileo

Same pattern as previous notebooks: connect to a Galileo project and log stream. This sets up the destination where all your agent traces will be stored. The console links let you view your traces in the Galileo web UI.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## Step 2: OBSERVE — Log a Multi-Turn Agent Session with Tool Calls

This is the **OBSERVE** pillar of Agent Control. We simulate a **customer support agent** that handles a billing dispute across 3 conversation turns. Each turn is a separate trace, but they are all grouped under one **session** (`billing-dispute-session-001`).

### How the 3 turns work:

| Turn | User Says | Agent Does | Spans Logged |
|------|-----------|------------|--------------|
| **1** | "I was charged twice" | Looks up billing history | agent + tool (lookup_billing) + LLM |
| **2** | "Process the refund" | Processes refund, sends email | agent + tool (process_refund) + tool (send_confirmation_email) + LLM |
| **3** | "Check my discount code" | Checks for active discounts | agent + tool (check_discounts) + LLM |

### Span tree structure:

```
Session: billing-dispute-session-001
├── Trace 1: "I was charged twice for my subscription"
│   ├── AgentSpan: BillingAgent
│   ├── ToolSpan: lookup_billing
│   └── LLMSpan: gpt-4o-mini
├── Trace 2: "Yes please, process the refund"
│   ├── AgentSpan: BillingAgent
│   ├── ToolSpan: process_refund
│   ├── ToolSpan: send_confirmation_email
│   └── LLMSpan: gpt-4o-mini
└── Trace 3: "Can you check if my discount code was applied?"
    ├── AgentSpan: BillingAgent
    ├── ToolSpan: check_discounts
    └── LLMSpan: gpt-4o-mini
```

Each `start_trace()` / `conclude()` pair creates one trace. The `start_session()` call ties them all together so Galileo can evaluate the full conversation.

In [ ]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_session(name='billing-dispute-session-001')

# Turn 1: Customer reports overcharge
logger.start_trace(input='I was charged twice for my subscription this month.')
logger.add_agent_span(
    input='I was charged twice for my subscription this month.',
    output="I'll look up your billing history and check for duplicate charges.",
    name='BillingAgent',
    duration_ns=100_000_000,
)
logger.add_tool_span(
    input='lookup_billing(customer_id="C-1234", months=1)',
    output='[{"date":"2026-03-01","amount":29.99,"desc":"Monthly Pro"},{"date":"2026-03-01","amount":29.99,"desc":"Monthly Pro"}]',
    name='lookup_billing',
    duration_ns=200_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='I was charged twice for my subscription.')],
    output=Message(role=MessageRole.assistant, content='I can see two charges of $29.99 on March 1st. This looks like a duplicate charge. Let me initiate a refund.'),
    model=MODEL,
    num_input_tokens=15,
    num_output_tokens=30,
    duration_ns=300_000_000,
)
logger.conclude(output='I can see two charges of $29.99 on March 1st. Let me initiate a refund.')

# Turn 2: Agent processes the refund
logger.start_trace(input='Yes please, process the refund.')
logger.add_agent_span(
    input='Yes please, process the refund.',
    output="I'll process the refund for the duplicate charge now.",
    name='BillingAgent',
    duration_ns=80_000_000,
)
logger.add_tool_span(
    input='process_refund(customer_id="C-1234", charge_id="CHG-5678", amount=29.99)',
    output='{"status":"approved","refund_id":"REF-9012","estimated_days":3}',
    name='process_refund',
    duration_ns=400_000_000,
)
logger.add_tool_span(
    input='send_confirmation_email(customer_id="C-1234", refund_id="REF-9012")',
    output='{"status":"sent","email":"customer@example.com"}',
    name='send_confirmation_email',
    duration_ns=150_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Process the refund.')],
    output=Message(role=MessageRole.assistant, content='Refund REF-9012 approved for $29.99. You should see it in 3 business days. A confirmation email has been sent.'),
    model=MODEL,
    num_input_tokens=10,
    num_output_tokens=25,
    duration_ns=250_000_000,
)
logger.conclude(output='Refund approved. Confirmation email sent.')

# Turn 3: Follow-up question
logger.start_trace(input='Can you also check if my discount code was applied?')
logger.add_agent_span(
    input='Can you also check if my discount code was applied?',
    output="I'll check your subscription for any active discount codes.",
    name='BillingAgent',
    duration_ns=90_000_000,
)
logger.add_tool_span(
    input='check_discounts(customer_id="C-1234")',
    output='{"active_discounts":[],"applied_to_last_charge":false}',
    name='check_discounts',
    duration_ns=180_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Check if my discount code was applied.')],
    output=Message(role=MessageRole.assistant, content="I don't see any active discount codes on your account. Would you like to add one?"),
    model=MODEL,
    num_input_tokens=12,
    num_output_tokens=20,
    duration_ns=200_000_000,
)
logger.conclude(output='No active discount codes found.')

logger.flush()
'Logged 3-turn billing dispute session'

## Step 3: OBSERVE — Log a Nested Workflow with Agent Decision Tree

Nested workflows let you represent **complex agent investigations** as a hierarchy of spans. In this example, a diagnostic agent investigates customer errors by:

1. Querying error logs
2. Checking the API configuration
3. Refreshing an expired API token
4. Summarizing the investigation

### Span tree structure:

```
Trace: "Investigate why customer C-5678 is seeing errors"
└── WorkflowSpan: ErrorInvestigationWorkflow
    └── AgentSpan: DiagnosticAgent
        ├── ToolSpan: query_error_logs
        ├── ToolSpan: check_api_config
        ├── ToolSpan: refresh_api_token
        └── LLMSpan: gpt-4o-mini (summarize findings)
```

The workflow span acts as a **container** that groups all the investigation steps. This makes it easy to see the entire pipeline as one logical unit in the Galileo Console.

In [ ]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_trace(input='Investigate why customer C-5678 is seeing errors.')

logger.add_workflow_span(
    input='Investigate customer C-5678 errors',
    output='Root cause identified: expired API token. Token refreshed and verified.',
    name='ErrorInvestigationWorkflow',
    duration_ns=1_200_000_000,
)

logger.add_agent_span(
    input='Investigate customer errors',
    output='I should check the error logs first, then verify the API configuration.',
    name='DiagnosticAgent',
    duration_ns=100_000_000,
)

logger.add_tool_span(
    input='query_error_logs(customer_id="C-5678", hours=24)',
    output='[{"error":"401 Unauthorized","endpoint":"/api/data","count":47,"first":"2026-03-14T10:00:00Z"}]',
    name='query_error_logs',
    duration_ns=300_000_000,
)

logger.add_tool_span(
    input='check_api_config(customer_id="C-5678")',
    output='{"api_key_status":"expired","last_rotated":"2026-01-15","expiry":"2026-03-14"}',
    name='check_api_config',
    duration_ns=250_000_000,
)

logger.add_tool_span(
    input='refresh_api_token(customer_id="C-5678")',
    output='{"new_token":"tk_abc123...","expires":"2026-06-14","status":"active"}',
    name='refresh_api_token',
    duration_ns=200_000_000,
)

logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Summarize the investigation.')],
    output=Message(role=MessageRole.assistant, content='Root cause: expired API token (expired 2026-03-14). 47 401 errors in 24h. Token refreshed, new expiry 2026-06-14.'),
    model=MODEL,
    num_input_tokens=20,
    num_output_tokens=30,
    duration_ns=300_000_000,
)

logger.conclude(output='Root cause identified and resolved: expired API token refreshed.')
logger.flush()
'Investigation workflow logged'

## Step 4: OBSERVE — Use @log Decorator for Agent Functions

The **`@galileo_log` decorator** is the most ergonomic way to instrument your code. Instead of manually calling `start_trace()`, `add_span()`, and `conclude()`, you simply decorate your existing Python functions.

### How it works:

- **`@galileo_log(span_type='tool')`** — wraps the function as a Tool Span
- **`@galileo_log(span_type='agent')`** — wraps the function as an Agent Span
- **`@galileo_log(span_type='workflow')`** — wraps the function as a Workflow Span

The decorator automatically captures the function's **input arguments** and **return value** as the span's input/output.

### Nested calls create nested spans automatically:

When `handle_customer_inquiry()` calls `customer_overview_agent()`, which calls `check_account_status()` and `get_recent_tickets()`, Galileo automatically nests the spans:

```
WorkflowSpan: handle_customer_inquiry
└── AgentSpan: customer_overview_agent
    ├── ToolSpan: check_account_status
    └── ToolSpan: get_recent_tickets
```

**This is the recommended approach for production code.** Decorate your existing functions and Galileo handles all the tracing plumbing for you.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

@galileo_log(span_type='tool')
def check_account_status(customer_id: str) -> str:
    return f'{{"customer_id":"{customer_id}","status":"active","plan":"pro","balance":0.00}}'

@galileo_log(span_type='tool')
def get_recent_tickets(customer_id: str) -> str:
    return f'[{{"id":"TK-101","subject":"Billing question","status":"resolved"}},{{"id":"TK-102","subject":"Feature request","status":"open"}}]'

@galileo_log(span_type='agent')
def customer_overview_agent(query: str) -> str:
    account = check_account_status(customer_id='C-1234')
    tickets = get_recent_tickets(customer_id='C-1234')
    return f'Account active (Pro plan). 2 recent tickets: 1 resolved, 1 open.'

@galileo_log(span_type='workflow')
def handle_customer_inquiry(user_request: str) -> str:
    return customer_overview_agent(user_request)

result = handle_customer_inquiry('Give me an overview of customer C-1234')
galileo_context.flush()
result

## Step 5: EVALUATE — Enable Core Agentic Metrics

This is the **EVALUATE** pillar of Agent Control. Galileo provides built-in metrics that automatically score your agent traces.

We start with the 3 core agentic metrics:

| Metric | What It Measures |
|--------|------------------|
| **`action_advancement_luna`** | Does each agent step make **meaningful progress** toward the goal? A high score means the agent is not wasting steps or going in circles. |
| **`agent_efficiency`** | How efficiently does the agent solve the task? Fewer unnecessary steps = higher score. An agent that resolves a billing dispute in 3 steps scores higher than one that takes 10. |
| **`agent_flow`** | Is the **sequence** of agent actions logical and coherent? For example, looking up billing before processing a refund is logical; processing a refund before looking anything up is not. |

After enabling these metrics, Galileo will score every new trace you log. You can view the scores in the Galileo Console alongside each trace.

In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.action_advancement_luna,
        GalileoMetrics.agent_efficiency,
        GalileoMetrics.agent_flow,
    ],
)

## Step 6: EVALUATE — Enable Tool Metrics

These metrics focus specifically on how well the agent uses its tools.

| Metric | What It Measures |
|--------|------------------|
| **`tool_error_rate`** | What fraction of tool calls result in errors? If the agent calls 10 tools and 3 fail, the error rate is 0.3. Lower is better. |
| **`tool_selection_quality`** | Did the agent choose the **right tool** for the task? For example, if the user asks about billing and the agent calls `check_discounts` instead of `lookup_billing`, this score will be low. |

> **Note:** Each call to `enable_metrics()` **replaces** the entire metric set for the log stream. So we include all previously enabled metrics plus the new ones.

In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.action_advancement_luna,
        GalileoMetrics.agent_efficiency,
        GalileoMetrics.agent_flow,
        GalileoMetrics.tool_error_rate,
        GalileoMetrics.tool_selection_quality,
    ],
)

## Step 7: EVALUATE — Enable Reasoning and Session Metrics

These metrics evaluate the agent's reasoning quality and the overall conversation experience. They are especially useful for multi-turn sessions (like the 3-turn billing dispute above).

| Metric | What It Measures |
|--------|------------------|
| **`reasoning_coherence`** | Is the agent's **chain of thought** logical? Does each reasoning step follow from the previous one? |
| **`action_completion_luna`** | Did the agent **successfully complete** the requested action? If the user asked for a refund and the agent processed it, this scores high. |
| **`conversation_quality`** | Overall quality of the multi-turn exchange. Considers helpfulness, clarity, and whether the agent stayed on topic. |
| **`user_intent_change`** | Did the user's goal **change** during the conversation? Detects if the user shifted topics mid-conversation, which may indicate confusion or poor agent guidance. |

With all 9 metrics enabled, you get a comprehensive view of agent quality across every dimension.

In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.action_advancement_luna,
        GalileoMetrics.agent_efficiency,
        GalileoMetrics.agent_flow,
        GalileoMetrics.tool_error_rate,
        GalileoMetrics.tool_selection_quality,
        GalileoMetrics.reasoning_coherence,
        GalileoMetrics.action_completion_luna,
        GalileoMetrics.conversation_quality,
        GalileoMetrics.user_intent_change,
    ],
)

## Step 8: PROTECT — Create an Agent Safety Guardrail Stage

This is the **PROTECT** pillar of Agent Control. Guardrails are runtime safety checks that can **block or override** unsafe agent outputs before they reach the user.

In this example, we create a guardrail stage that checks the **toxicity** of agent outputs. If the `input_toxicity` score is >= 0.5, the guardrail overrides the response with a safe message.

**How it works:**
- A **Rule** defines the condition: `input_toxicity >= 0.5`
- A **Ruleset** pairs the rule with an **OverrideAction** (the safe replacement message)
- A **Stage** is the deployable unit that you invoke at runtime

The try/get/update/except/create pattern makes this cell **idempotent** — you can run it multiple times without errors.

In [ ]:
agent_safety_rulesets = [
    Ruleset(
        rules=[Rule(metric='input_toxicity', operator=RuleOperator('gte'), target_value=0.5)],
        action=OverrideAction(choices=['Agent response blocked: content policy violation detected.']),
    ),
]

try:
    get_protect_stage(project_name=PROJECT_NAME, stage_name='agent-safety-gate')
    update_protect_stage(
        project_name=PROJECT_NAME,
        stage_name='agent-safety-gate',
        prioritized_rulesets=agent_safety_rulesets,
    )
except NotFoundError:
    create_protect_stage(
        name='agent-safety-gate',
        project_name=PROJECT_NAME,
        stage_type=StageType.central,
        prioritized_rulesets=agent_safety_rulesets,
    )

'Agent safety guardrail stage created'

## Step 9: PROTECT — Test the Agent Safety Guardrail

Now we test the guardrail with two agent outputs:

1. **Safe output** — A professional refund confirmation message. This should return `not_triggered`, meaning the guardrail lets it through.
2. **Unsafe output** — A hostile, unprofessional response. This should return `triggered`, meaning the guardrail blocks it and replaces it with the safe override message.

In a real application, you would check the `status` field after every agent response and either:
- **Not triggered** — send the agent's response to the user
- **Triggered** — send the override message instead

In [ ]:
safe_result = invoke_protect(
    payload=Payload(input='Your refund of $29.99 has been processed and will appear in 3 business days.'),
    stage_name='agent-safety-gate',
    project_name=PROJECT_NAME,
)

unsafe_result = invoke_protect(
    payload=Payload(input='You are an idiot for not reading the terms of service. Your request is denied.'),
    stage_name='agent-safety-gate',
    project_name=PROJECT_NAME,
)

{
    'safe_status': str(getattr(safe_result, 'status', safe_result)),
    'unsafe_status': str(getattr(unsafe_result, 'status', unsafe_result)),
    'unsafe_action': getattr(unsafe_result, 'action_result', None),
}

## Step 10: OBSERVE — Circular Tool Detection Pattern

A common failure mode for AI agents is **infinite tool loops** — the agent keeps calling the same tools repeatedly without making progress. For example, a search agent that finds no results might keep refining and re-searching forever.

This pattern demonstrates how to detect and break such cycles:

1. **Track tool call history** in a list
2. **Check a sliding window** of recent calls for repetition
3. **Break the loop** when only 1-2 unique calls appear in the last N attempts

The `MAX_TOOL_CALLS` constant sets an absolute upper limit, and `LOOP_DETECTION_WINDOW` defines how many recent calls to check for repetition.

**Why this matters for production agents:** Without loop detection, a single stuck conversation can consume unlimited API credits and time. This is a reliability pattern that every production agent should implement.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

MAX_TOOL_CALLS = 5
LOOP_DETECTION_WINDOW = 4

@galileo_log(span_type='tool')
def fetch_data(query: str) -> str:
    return f'{{"results": [], "message": "No results for: {query}"}}'

@galileo_log(span_type='tool')
def refine_query(original: str) -> str:
    return f'refined: {original}'

@galileo_log(span_type='agent')
def resilient_search_agent(query: str) -> str:
    tool_history = []
    current_query = query

    for i in range(MAX_TOOL_CALLS):
        result = fetch_data(current_query)
        tool_history.append(f'fetch_data({current_query})')

        if '"results": []' not in result:
            return f'Found results on attempt {i+1}: {result}'

        # Circular tool detection: check if last N calls are repeating
        if len(tool_history) >= LOOP_DETECTION_WINDOW:
            recent = tool_history[-LOOP_DETECTION_WINDOW:]
            if len(set(recent)) <= 2:  # only 1-2 unique calls = loop
                return f'Loop detected after {i+1} attempts. Stopping. Last query: {current_query}'

        current_query = refine_query(current_query)
        tool_history.append(f'refine_query({current_query})')

    return f'Max attempts ({MAX_TOOL_CALLS}) reached without results.'

result = resilient_search_agent('Find customer C-9999 billing history')
galileo_context.flush()
result

## Step 11: Clean Up

Delete the project to remove all stages, log streams, and traces created during this demo. This keeps your Galileo workspace tidy.

In [ ]:
delete_project(name=PROJECT_NAME)